# Previsão do preço do Bitcoin com LSTM — Relatório de experimentos (busca em blocos)

**Objetivo:** descobrir o que **realmente** melhora o modelo quando cada hiperparâmetro muda, separando efeito de ruído.
Para isso o estudo mede o ruído entre seeds (Bloco 0), estima o efeito de cada eixo controlando os demais, reconstrói a
cadeia de decisões e termina com uma ablação do campeão.

**Metodologia** (ver `METODOLOGIA.md`). Em vez de escolher um valor por vez ou cruzar tudo num grid gigante, o estudo é
dividido em **blocos sequenciais**, cada um respondendo a uma pergunta. O campeão de um bloco é lido automaticamente pelo
bloco seguinte, sem nenhum valor escolhido à mão. Como cada bloco varia só um grupo coeso de hiperparâmetros, o espaço
total explorado pode ser bem maior que o de um grid puro. Blocos com muitos eixos usam **busca aleatória** (n combinações
distintas sorteadas do produto cartesiano, com seed fixa), lida pelo efeito marginal de cada eixo.

| Bloco | Pergunta | Espaço de busca |
|---|---|---|
| Bloco 0 — Referência | Onde partimos? Quanto as previsões ingênuas e um LSTM com os valores iniciais recomendados acertam? | passeio_aleatorio, media_historica, ultimo_retorno, regressao_linear, lstm_padrao, lstm_padrao_seed1, lstm_padrao_seed2, lstm_padrao_seed3, lstm_padrao_seed4 |
| Bloco 1 — Representação da entrada | O que a rede deve ver: quantos dias de histórico, quais features e qual alvo? | lookback × features × target |
| Bloco 2 — Camadas ocultas, nós e camada densa | Quantas camadas LSTM ocultas, quantos nós por camada e quantas unidades densas absorvem o problema? | num_layers × hidden_size × fc_neurons (aleatória, 50) |
| Bloco 3 — Funções de ativação e inicialização dos pesos | Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã? | lstm_activation × activation × weight_init (aleatória, 40) |
| Bloco 4 — Otimização: taxa de aprendizagem, decaimento, momentum e batch | Qual algoritmo, taxa de aprendizagem, taxa de decaimento, momentum e tamanho de batch fazem a rede convergir melhor? | optimizer × lr × decay_rate × momentum × batch_size (aleatória, 60) |
| Checagem de interação | Com a nova otimização, a estrutura campeã do Bloco 2 continua sendo a melhor? | 2, 3º de `lstm_b2_capacidade` com a config. atual |
| Bloco 5 — Dropout e decaimento dos pesos | Com a rede convergindo bem, quanto dropout e quanta penalização dos pesos controlam o overfitting ao ruído do mercado? | dropout × rnn_dropout × input_dropout × weight_decay (aleatória, 50) |
| Complementar A — Camadas e nós sob regularização | Com o dropout ajustado, redes com mais nós ou mais camadas voltam a compensar? | num_layers × hidden_size |
| Complementar B — Número de épocas (early stopping) | O resultado está limitado pelo orçamento de épocas? Quanta paciência o early stopping deve ter? | patience |
| Bônus — LSTM × GRU | A célula LSTM é necessária, ou uma GRU (menos portas, menos parâmetros) chega ao mesmo resultado? | cell |
| Bônus — Data augmentation | Criar variações plausíveis das janelas de treino (ruído, amplitude, velocidade, ordem, recorte) melhora a previsão do campeão? | augment × aug_strength |
| Ablação — o que realmente importa no campeão | Das mudanças que levaram do LSTM padrão ao campeão, quais realmente melhoram o modelo e quais são dispensáveis? | desfaz, uma de cada vez, cada mudança do campeão em relação ao padrão |

**Dados.** Série(s): `BTC` (`data-bitcoin_timedata-2023_v2 - data-bitcoin_timedata-2023_v2.csv`), de 2017-08-17 a 2023-08-01 (fonte `csv`, preço `Close`).
Com várias séries, um único modelo é treinado com as janelas de todas elas (normalização por série) e as métricas saem no
geral e **por série**. Os preços ficam congelados em `data/precos/`, com o MD5 no `parametros.json` de cada experimento.

**Protocolo de avaliação.**
- **Validação walk-forward em 5 folds** (`expanding`, blocos de 180 dias): o fold k treina em tudo antes do bloco de validação k e valida nele. As partições são por data e são as mesmas em todos os experimentos (comparações pareadas). Um embargo de h dias impede que alvos do treino entrem na validação.
- **Métrica de decisão:** `val/rmse` (menor é melhor). Além dela são registradas, no geral e por ação: `mse`, `rmse` e `mae` (no log-retorno de h dias), `mape` (% no preço), `theil` (U de Theil: erro do modelo ÷ erro do passeio aleatório, < 1 = bate a referência), **`pocid`** (Prediction Of Change In Direction, %: acerto na previsão de subida/queda da cotação, com D_t = 1 se (P_t − P_t−1)(P̂_t − P̂_t−1) > 0), `da` (acurácia direcional em relação ao preço atual, %), `skill`, `r2` e `ic`. As métricas são comparáveis entre alvos (retorno ou preço).
- **Triagem:** cada configuração roda os folds [3, 4, 5]; as 3 melhores **completam os demais folds** (sem refazer nada) e o campeão é a maior média nos 5 folds, nunca a média parcial da triagem (maldição do vencedor).
- **Escolha sempre pela validação.** O teste (a partir de 2022-08-01) só é revelado na seção final, para os campeões e as referências.
- Diferenças menores que o desvio entre folds não devem ser tratadas como melhora real: use a comparação pareada.

**Registro dos resultados.** Cada experimento grava em `outputs/{exp_name}/`:

| Arquivo | Conteúdo |
|---|---|
| `parametros.json` | hiperparâmetros exatos, arquitetura, partições, MD5 dos dados, versões e commit |
| `historico_treino.csv` | uma linha por **fold × época**: loss e métricas de treino/validação (gerais e por ação), gap |
| `historico_treino_agregado.csv` | média e desvio entre folds, por época |
| `resultados.json` | por fold: validação na melhor época e teste do modelo restaurado; médias |
| `melhor_modelo.pth` | pesos do melhor fold; cada fold em `folds/fold_k/` (com `predicoes_val.csv` e `predicoes_test.csv`) |

Cada bloco grava em `outputs/_grids/{bloco}/` (`configs.csv`, `descartadas.csv`, `ranking_triagem.csv`, `ranking_final.csv`,
`campeao.json`, `params/`, `logs/`). Tabelas e figuras ficam em `outputs/_relatorio/`. Tudo é gravado a cada época **em disco
primeiro**; W&B e GitHub são espelhos opcionais.

**Tempo estimado:** cerca de 1.000 treinos de fold. Medido numa GPU de notebook (RTX 3050): um fold do LSTM padrão leva
3–13 s, e o Bloco 0 inteiro (9 configurações × 5 folds) leva 1,5 min. Estimativa para o estudo completo no Kaggle com 2× T4:
**~1–3 h**. O Bloco 3 é o mais lento, porque ativações ≠ tanh usam uma célula LSTM própria, e os blocos com mais épocas (5, 7)
também demoram mais. Se o limite de 12 h estourar, basta retomar (veja abaixo).

## Como executar

O notebook é o mesmo em qualquer ambiente: ele clona o código do GitHub (ou usa a cópia local), acha os dados onde
estiverem e pula o que não estiver disponível. **Nada além de Python + GPU é obrigatório** (W&B e GitHub são opcionais).

### No Kaggle (execução principal)

1. **Dataset:** *Datasets → New Dataset*, envie o CSV do Bitcoin (`data-bitcoin_timedata-2023_v2 - ….csv`).
2. **Notebook:** *Code → New Notebook → File → Import Notebook* e escolha `Kaggle_LSTM.ipynb`.
3. **Input:** *Add Input* → o dataset do passo 1. O notebook acha o CSV sozinho em `/kaggle/input` (pelo nome, ou o
   único CSV com colunas `date`/`close`).
4. **Secrets:** *Add-ons → Secrets* → `GITHUB_TOKEN` e `WANDB_API_KEY` (os mesmos valores do `.env`), marcados como anexados.
5. **Settings:** *Accelerator* **GPU T4 ×2**, *Internet* **On** (clone do código, W&B e envio dos resultados).
6. *Save Version → **Save & Run All (Commit)***. A execução roda em segundo plano (limite de 12 h por sessão); a saída
   fica em *Output* (`outputs/` e `outputs.zip`) e, com o token, no branch `resultados` do GitHub.

### No Colab ou Jupyter local

| Item | Colab | Jupyter local |
|---|---|---|
| GPU | *Runtime → Change runtime type → GPU* | a que o PyTorch enxergar (CPU funciona, mais devagar) |
| Código | clonado de `REPO_URL` | abra o notebook **dentro** do repositório |
| Dados | CSV em `/content`, ou `DATA_PATH` | o CSV na raiz do repositório, ou `DATA_PATH` |
| Tokens | *Colab Secrets* (mesmos nomes) | arquivo `.env` na raiz (modelo em `.env.example`) |
| Saída | `/content/outputs` | `outputs/` no repositório |

**W&B.** Uma run por configuração (grupo = bloco, `config` = hiperparâmetros, resumo = médias entre folds). Os painéis
*Parameter importance* e *Parallel coordinates* do projeto mostram o mesmo que as análises deste notebook.
O teste nunca é enviado ao W&B (só aparece na seção final).

**Resultados no GitHub.** Com `GITHUB_TOKEN`, ao fim de cada bloco os CSV/JSON/PNG de `outputs/` vão para o branch
`RESULTS_BRANCH`, na pasta `RUN_NAME/outputs/` (branch órfão, separado do código; `.pth` só com `SYNC_PESOS = True`).

**Retomada.** Todo experimento pula os folds já concluídos. Para continuar uma execução interrompida (ex.: limite de 12 h):
`RUN_NAME` igual ao da execução anterior e `RESUME_FROM = "github"`, ou `RESUME_FROM = "/kaggle/input/<output anterior>/outputs"`.

## 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
import os

REPO_URL = "https://github.com/diegoflyra/if702-miniproject-2.git"  # repositório com src/, grids/, config/
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfão só de resultados
RUN_NAME = ""                    # vazio = "lstm-AAAAMMDD-HHMM"; fixe um nome para retomar pelo GitHub
RESUME_FROM = ""                 # "" | "github" | pasta outputs de uma execução anterior
WORKERS_PER_GPU = 2              # experimentos simultâneos por GPU (LSTMs pequenos; o Kaggle tem 4 CPUs)
CPU_WORKERS = 1                  # sem GPU
DATA_PATH = ""                   # CSV (ou pasta) dos dados; vazio = raiz do repo → /kaggle/input → /content
SYNC_PESOS = False               # enviar também os .pth ao GitHub
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "if702-miniproject-2-lstm")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")

In [ ]:
import os
import shutil
import subprocess
import sys
import time


def _load_dotenv(path=".env"):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'"")


_load_dotenv()


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(path):
    return all(os.path.exists(os.path.join(path, p)) for p in ("src/grid_search.py", "grids", "config/estudo.json"))


GITHUB_TOKEN = _secret("GITHUB_TOKEN")
here = os.getcwd()
if _is_repo(here):
    REPO_DIR = here
    print(f"Código: cópia local em {REPO_DIR}")
else:
    REPO_DIR = "/tmp/lstm-acoes"
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    if _is_repo(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        cloned = subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR])
        if cloned.returncode != 0 or not _is_repo(REPO_DIR):
            raise RuntimeError("Falha ao clonar o repositório. Repositório privado exige GITHUB_TOKEN "
                               "(Kaggle Secrets, Colab Secrets ou variável de ambiente), ou abra o notebook "
                               "a partir de uma cópia local do projeto (pasta com src/, grids/ e config/).")
    print(f"Código: {REPO_DIR}")

os.chdir(REPO_DIR)
%cd {REPO_DIR}
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
import importlib.util

skip, pkgs = [], []
with open("requirements.txt", encoding="utf-8") as f:
    for line in f:
        pkg = line.strip()
        if not pkg or pkg.startswith("#"):
            continue
        name = pkg.split("==")[0].split(">=")[0].split("<=")[0].split("~=")[0].split("[")[0].strip().lower()
        if name == "torch" and importlib.util.find_spec(name) is not None:
            skip.append(name)
            continue
        pkgs.append(pkg)
if skip:
    print("Já instalado, não reinstalar (preserva CUDA do ambiente):", ", ".join(skip))
if pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import pandas as pd

if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = REPO_DIR
OUTPUTS = os.path.join(WORKDIR, "outputs")
os.environ["EXP_OUTPUT_DIR"] = OUTPUTS
os.makedirs(OUTPUTS, exist_ok=True)
print(f"Resultados em {OUTPUTS}")

RUN_NAME = RUN_NAME or time.strftime("lstm-%Y%m%d-%H%M")
os.environ.update({"RUN_NAME": RUN_NAME, "RESULTS_REPO_URL": RESULTS_REPO_URL, "RESULTS_BRANCH": RESULTS_BRANCH,
                   "GITHUB_TOKEN": GITHUB_TOKEN, "WANDB_PROJECT": WANDB_PROJECT, "WANDB_ENTITY": WANDB_ENTITY})
if DATA_PATH:
    os.environ["DATA_PATH"] = DATA_PATH
print(f"RUN_NAME = {RUN_NAME}")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import data as data_mod
import github_sync
import report_utils as rep

if RESUME_FROM == "github":
    github_sync.pull(OUTPUTS)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, OUTPUTS, dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

wandb_key = _secret("WANDB_API_KEY")
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ.pop("WANDB_MODE", None)
    print(f"W&B: chave carregada (projeto {WANDB_PROJECT}).")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado (sem chave). Resultados continuam em {OUTPUTS}.")

if not (GITHUB_TOKEN and RESULTS_REPO_URL):
    print("GitHub: sem GITHUB_TOKEN/RESULTS_REPO_URL, os resultados ficam só em disco (e em outputs.zip).")


def backup(bloco=""):
    """Backup parcial ao fim de cada bloco: outputs.zip (sem .pth) + espelho no GitHub, se configurado."""
    subprocess.run(f"cd {WORKDIR} && zip -qr outputs.zip outputs -x '*.pth' '*.tmp'", shell=True, check=False)
    if os.path.isfile(os.path.join(WORKDIR, "outputs.zip")):
        print(f"outputs.zip: {os.path.getsize(os.path.join(WORKDIR, 'outputs.zip')) / 1e6:.1f} MB")
    github_sync.push(OUTPUTS, weights=SYNC_PESOS, message=f"{RUN_NAME}: {bloco or 'backup'}")

### Dados e partições

Preços congelados por ação (MD5 no manifesto) e as datas de cada fold walk-forward. O teste aparece só como período; nenhum número dele é usado até o fim.

In [ ]:
manifesto = data_mod.prepare_prices()
display(data_mod.describe_folds())

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino): GPUs, dados sem vazamento, modelos e todos os grids
if shutil.which("nvidia-smi"):
    !nvidia-smi -L
else:
    print("nvidia-smi não encontrado; o treino usa o device que o PyTorch enxergar (CPU se não houver GPU).")
!python tests/check_data.py
!python tests/check_models.py
!python tests/check_metrics.py
!python tests/check_augment.py
!python tests/check_grids.py

## Bloco 0 — Referência

**Pergunta:** Onde partimos? Quanto as previsões ingênuas e um LSTM com os valores iniciais recomendados acertam?

Em séries financeiras, a referência difícil de bater é o **passeio aleatório** (prever que o preço de amanhã é o de hoje): `theil` < 1 e `skill` > 0 significam batê-lo, e a **POCID** mostra se o modelo acerta a direção da cotação. Também entram a média histórica, o último retorno (momentum ingênuo), uma **regressão linear** sobre a mesma janela (ser recorrente ajuda?) e o **LSTM padrão**, com os pontos de partida da lista: 1 camada oculta de 50 nós, densa de 10 unidades com ReLU, dropout de 20%, tanh na célula, Adam com lr 0,001 e decaimento 0,97 por época. Ele é a base do Bloco 1. **Ruído entre seeds:** o LSTM padrão é treinado com 5 seeds (42, 1, 2, 3, 4). A diferença entre duas execuções que só mudam a seed é a régua do estudo: uma mudança de hiperparâmetro só "melhora de verdade" se o ganho passar de ~2× esse ruído e se repetir na maioria dos folds.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Configuração | Parâmetros |
|---|---|
| `passeio_aleatorio` | `model=naive_zero` |
| `media_historica` | `model=naive_mean` |
| `ultimo_retorno` | `model=naive_last` |
| `regressao_linear` | `model=linear` |
| `lstm_padrao` | `model=lstm` |
| `lstm_padrao_seed1` | `model=lstm`, `seed=1` |
| `lstm_padrao_seed2` | `model=lstm`, `seed=2` |
| `lstm_padrao_seed3` | `model=lstm`, `seed=3` |
| `lstm_padrao_seed4` | `model=lstm`, `seed=4` |

**9 configurações.** Todas as configurações rodam os 5 folds.

**O que observar:** Se o LSTM padrão bate o passeio aleatório (`val/theil` < 1) e se a POCID passa de 50% de forma consistente entre folds. E o tamanho do ruído entre seeds, que define o menor efeito detectável.

In [ ]:
!python src/grid_search.py grids/lstm_b0_referencia.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

In [ ]:
rep.discarded_configs("lstm_b0_referencia")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.plot_grid_bars("lstm_b0_referencia")

#### Confirmação das finalistas e campeão — `lstm_b0_referencia`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b0_referencia", "final")

In [ ]:
rep.show_champion("lstm_b0_referencia")
rep.plot_finalists("lstm_b0_referencia")

#### Ruído entre seeds

A régua do estudo: o menor efeito que se distingue de sorte na inicialização.

In [ ]:
ruido = rep.noise_floor()
rep.noise_floor("val/pocid")

### 📝 Análise — Bloco 0

- **Algum modelo bateu o passeio aleatório (theil < 1, skill > 0)?** _…_
- **O LSTM padrão superou a regressão linear na mesma janela?** _…_
- **POCID e acurácia direcional: acima de 50% de forma consistente?** _…_
- **Ruído entre seeds: qual o menor efeito que o estudo consegue detectar?** _…_

In [ ]:
backup("lstm_b0_referencia")

## Bloco 1 — Representação da entrada

**Pergunta:** O que a rede deve ver: quantos dias de histórico, quais features e qual alvo?

Não está na lista de hiperparâmetros da rede, mas vem antes dela: a janela e as features definem o problema que o LSTM resolve. O alvo `close` (preço normalizado) é comparado com `log_return`; as métricas são sempre calculadas da mesma forma, então os dois são comparáveis.

**Base:** `config/estudo.json → padrao` (o LSTM padrão).

| Eixo | Valores |
|---|---|
| `lookback` | `5`, `10`, `20`, `40`, `60`, `120` |
| `features` | `retornos`, `retornos_volume`, `ohlcv`, `tecnicos` |
| `target` | `log_return`, `close` |

**48 combinações.** Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se janelas longas ajudam ou só trazem ruído, se volume, amplitude e indicadores acrescentam algo sobre só retornos, e se o alvo em preço extrapola mal fora da faixa do treino.

In [ ]:
!python src/grid_search.py grids/lstm_b1_entrada.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b1_entrada`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b1_entrada", "triagem")

In [ ]:
rep.discarded_configs("lstm_b1_entrada")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b1_entrada`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b1_entrada")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b1_entrada", row="lookback", col="features", facet="target")

In [ ]:
rep.heatmap("lstm_b1_entrada", row="lookback", col="features", facet="target", value="val/pocid_mean")

In [ ]:
rep.param_effect("lstm_b1_entrada")

In [ ]:
rep.plot_grid_bars("lstm_b1_entrada")

#### Confirmação das finalistas e campeão — `lstm_b1_entrada`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b1_entrada", "final")

In [ ]:
rep.show_champion("lstm_b1_entrada")
rep.plot_finalists("lstm_b1_entrada")

### 📝 Análise — Bloco 1

- **Janela: qual tamanho foi útil?** _…_
- **Features: o que acrescentou sobre só retornos?** _…_
- **Alvo log_return × close:** _…_
- **Diferença do campeão para o LSTM padrão (validação):** _…_

In [ ]:
backup("lstm_b1_entrada")

## Bloco 2 — Camadas ocultas, nós e camada densa

**Pergunta:** Quantas camadas LSTM ocultas, quantos nós por camada e quantas unidades densas absorvem o problema?

Itens 1 e 2 da lista. Regra prática: 1 camada oculta basta para problemas simples e 2 para os razoavelmente complexos; muitos nós aumentam a capacidade (com regularização) e poucos causam subajuste. Na camada densa, 5–10 unidades são um bom ponto de partida (`[]` = sem densa; `[10, 10]` = duas densas). Espaço de 3 × 7 × 7 = 147 combinações, com **busca aleatória** de 50; o dropout de 20% do padrão é mantido.

**Base:** o melhor campeão entre `lstm_b1_entrada`.

| Eixo | Valores |
|---|---|
| `num_layers` | `1`, `2`, `3` |
| `hidden_size` | `8`, `16`, `32`, `50`, `64`, `128`, `256` |
| `fc_neurons` | `[]`, `[5]`, `[10]`, `[25]`, `[50]`, `[10, 10]`, `[25, 10]` |

**147 combinações possíveis; busca aleatória de 50** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds. Limite: 2,000,000 parâmetros.

**O que observar:** Onde a capacidade satura, se mais camadas só aumentam o gap treino–validação, e se a densa ajuda ou atrapalha.

In [ ]:
rep.show_champion("lstm_b1_entrada")

In [ ]:
!python src/grid_search.py grids/lstm_b2_capacidade.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b2_capacidade`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b2_capacidade", "triagem")

In [ ]:
rep.discarded_configs("lstm_b2_capacidade")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b2_capacidade`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b2_capacidade")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b2_capacidade", row="hidden_size", col="num_layers")

In [ ]:
rep.heatmap("lstm_b2_capacidade", row="hidden_size", col="num_layers", value="gap/rmse_mean")

In [ ]:
rep.heatmap("lstm_b2_capacidade", row="fc_neurons", col="num_layers")

In [ ]:
rep.param_effect("lstm_b2_capacidade")

In [ ]:
rep.plot_grid_bars("lstm_b2_capacidade")

#### Confirmação das finalistas e campeão — `lstm_b2_capacidade`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b2_capacidade", "final")

In [ ]:
rep.show_champion("lstm_b2_capacidade")
rep.plot_finalists("lstm_b2_capacidade")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b2_capacidade"), "lstm_b2_capacidade__*")

### 📝 Análise — Bloco 2

- **Camadas ocultas: 1, 2 ou 3?** _…_
- **Nós por camada: onde satura? Houve subajuste com poucos nós?** _…_
- **Camada densa (unidades):** _…_
- **Ganho sobre o Bloco 1 (validação):** _…_

In [ ]:
backup("lstm_b2_capacidade")

## Bloco 3 — Funções de ativação e inicialização dos pesos

**Pergunta:** Quais ativações (na célula LSTM e nas densas) e qual esquema de inicialização funcionam melhor com a estrutura campeã?

Itens 4 e 6 da lista. **Célula LSTM:** as portas são sempre sigmoid; varia a ativação da candidata e da saída (tanh é o padrão; sigmoid, softsign e ReLU usam uma célula própria, mais lenta que o cuDNN). **Densas:** ReLU, tanh, sigmoid e ELU; se a campeã não tiver camada densa, esse eixo não tem efeito e as combinações equivalentes são descartadas. A saída é sempre linear, porque o alvo é contínuo (sigmoid e softmax na saída são para classificação). **Inicialização:** pesos pequenos e aleatórios; `padrao` = PyTorch U(±1/√h), `uniforme` = U(±0,05), `normal` = N(0; 0,05), `xavier` (Glorot), `glorot_ortogonal` (padrão do Keras: ortogonal na recorrência e bias de esquecimento 1) e `he` (Kaiming, pensado para ReLU). Espaço de 4 × 4 × 6 = 96 combinações, com busca aleatória de 40.

**Base:** o melhor campeão entre `lstm_b2_capacidade`.

| Eixo | Valores |
|---|---|
| `lstm_activation` | `tanh`, `sigmoid`, `softsign`, `relu` |
| `activation` | `relu`, `tanh`, `sigmoid`, `elu` |
| `weight_init` | `padrao`, `uniforme`, `normal`, `xavier`, `glorot_ortogonal`, `he` |

**96 combinações possíveis; busca aleatória de 40** (seed 0). A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** Se a ativação da célula muda algo além de tanh (ReLU pode explodir sem limite), e se a inicialização afeta a convergência (`melhor_epoca_media`) e a variância entre folds.

In [ ]:
rep.show_champion("lstm_b2_capacidade")

In [ ]:
!python src/grid_search.py grids/lstm_b3_ativacao_init.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b3_ativacao_init`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b3_ativacao_init", "triagem")

In [ ]:
rep.discarded_configs("lstm_b3_ativacao_init")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b3_ativacao_init`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b3_ativacao_init")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b3_ativacao_init", row="weight_init", col="lstm_activation")

In [ ]:
rep.heatmap("lstm_b3_ativacao_init", row="weight_init", col="lstm_activation", value="melhor_epoca_media")

In [ ]:
rep.heatmap("lstm_b3_ativacao_init", row="activation", col="lstm_activation")

In [ ]:
rep.param_effect("lstm_b3_ativacao_init")

In [ ]:
rep.plot_grid_bars("lstm_b3_ativacao_init")

#### Confirmação das finalistas e campeão — `lstm_b3_ativacao_init`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b3_ativacao_init", "final")

In [ ]:
rep.show_champion("lstm_b3_ativacao_init")
rep.plot_finalists("lstm_b3_ativacao_init")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b3_ativacao_init"), "lstm_b3_ativacao_init__*")

### 📝 Análise — Bloco 3

- **Ativação da célula LSTM:** _…_
- **Ativação das densas:** _…_
- **Inicialização: afetou a convergência ou só o ruído entre folds?** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
backup("lstm_b3_ativacao_init")

## Bloco 4 — Otimização: taxa de aprendizagem, decaimento, momentum e batch

**Pergunta:** Qual algoritmo, taxa de aprendizagem, taxa de decaimento, momentum e tamanho de batch fazem a rede convergir melhor?

Item 7 (taxa de aprendizagem, entre 0 e 0,1, de preferência com decaimento) e item 5 (taxa de decaimento, com ponto de partida 0,97). Aqui o **decaimento é aplicado à taxa de aprendizagem**: lr ← lr × `decay_rate` a cada época, e 1,0 = sem decaimento. O decaimento dos **pesos** (penalização L2, `weight_decay`) é regularização e fica no Bloco 5. Também entram momentum (SGD e RMSprop; o Adam não usa) e tamanho de batch. O grid de lr é comum a todos os algoritmos de propósito, para mostrar a faixa útil de cada um. Espaço de 3 × 7 × 5 × 4 × 5 = 2.100 combinações, com **busca aleatória** de 60; combinações equivalentes (momentum no Adam) são descartadas sem gastar o orçamento.

**Base:** o melhor campeão entre `lstm_b3_ativacao_init`.

| Eixo | Valores |
|---|---|
| `optimizer` | `sgd`, `adam`, `rmsprop` |
| `lr` | `0.0001`, `0.0003`, `0.001`, `0.003`, `0.01`, `0.03`, `0.1` |
| `decay_rate` | `1`, `0.99`, `0.97`, `0.95`, `0.9` |
| `momentum` | `0`, `0.5`, `0.9`, `0.99` |
| `batch_size` | `16`, `32`, `64`, `128`, `256` |

**2100 combinações possíveis; busca aleatória de 60** (seed 0). Fixos no bloco: `scheduler=exponential`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** A faixa útil de lr de cada otimizador, onde ele diverge (lr 0,1), se o decaimento estabiliza lr altas, e se batch pequeno (gradiente mais ruidoso) regulariza.

In [ ]:
rep.show_champion("lstm_b3_ativacao_init")

In [ ]:
!python src/grid_search.py grids/lstm_b4_otimizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b4_otimizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b4_otimizacao", "triagem")

In [ ]:
rep.discarded_configs("lstm_b4_otimizacao")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b4_otimizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b4_otimizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b4_otimizacao", row="optimizer", col="lr")

In [ ]:
rep.heatmap("lstm_b4_otimizacao", row="decay_rate", col="lr")

In [ ]:
rep.heatmap("lstm_b4_otimizacao", row="optimizer", col="lr", value="melhor_epoca_media")

In [ ]:
rep.param_effect("lstm_b4_otimizacao")

In [ ]:
rep.plot_grid_bars("lstm_b4_otimizacao")

#### Confirmação das finalistas e campeão — `lstm_b4_otimizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b4_otimizacao", "final")

In [ ]:
rep.show_champion("lstm_b4_otimizacao")
rep.plot_finalists("lstm_b4_otimizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b4_otimizacao"), "lstm_b4_otimizacao__*")

### 📝 Análise — Bloco 4

- **Faixa de lr útil de cada otimizador:** _…_
- **Houve divergência?** _…_
- **Decaimento da lr: 0,97 é um bom valor? Interage com a lr?** _…_
- **Momentum e batch size:** _…_
- **Ganho sobre o Bloco 3 (validação):** _…_

In [ ]:
backup("lstm_b4_otimizacao")

## Checagem de interação

**Pergunta:** Com a nova otimização, a estrutura campeã do Bloco 2 continua sendo a melhor?

O 2º e o 3º colocados do Bloco 2 (camadas, nós e densa) são treinados com a ativação, a inicialização e a otimização campeãs (K folds). O Bloco 5 parte da melhor rede entre o campeão do Bloco 4 e esta checagem.

**Base:** o melhor campeão entre `lstm_b4_otimizacao`.

Todas as configurações rodam os 5 folds.

**O que observar:** Se a ordem das estruturas se inverte com a nova otimização.

In [ ]:
rep.show_champion("lstm_b4_otimizacao")

In [ ]:
!python src/grid_search.py grids/lstm_b4_checagem.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

In [ ]:
rep.discarded_configs("lstm_b4_checagem")  # combinações descartadas antes do treino e o motivo

#### Confirmação das finalistas e campeão — `lstm_b4_checagem`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b4_checagem", "final")

In [ ]:
rep.show_champion("lstm_b4_checagem")
rep.plot_finalists("lstm_b4_checagem")

In [ ]:
pd.concat([rep.grid_ranking("lstm_b4_otimizacao", "final").head(1),
           rep.grid_ranking("lstm_b4_checagem", "final")], ignore_index=True)

### 📝 Análise — Checagem de interação

- **A ordem das estruturas se manteve com a nova otimização?** _…_

In [ ]:
backup("lstm_b4_checagem")

## Bloco 5 — Dropout e decaimento dos pesos

**Pergunta:** Com a rede convergindo bem, quanto dropout e quanta penalização dos pesos controlam o overfitting ao ruído do mercado?

Item 3 da lista, mais o decaimento dos pesos do item 5. Todo LSTM é acompanhado de dropout: `rnn_dropout` entre camadas LSTM empilhadas (só vale com 2+ camadas; com 1 camada é equivalente a 0 e é descartado), `dropout` depois da última LSTM e entre as densas (nunca na saída), e `input_dropout` na entrada. O recomendado é partir de 20% e não passar de 50%. `weight_decay` é a penalização L2, que encolhe os pesos a cada atualização. Espaço de 6 × 6 × 3 × 5 = 540 combinações, com busca aleatória de 50. Mais épocas e paciência, porque a regularização atrasa a convergência.

**Base:** o melhor campeão entre `lstm_b4_otimizacao`, `lstm_b4_checagem`.

| Eixo | Valores |
|---|---|
| `dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.4`, `0.5` |
| `rnn_dropout` | `0`, `0.1`, `0.2`, `0.3`, `0.4`, `0.5` |
| `input_dropout` | `0`, `0.1`, `0.2` |
| `weight_decay` | `0`, `1e-06`, `1e-05`, `0.0001`, `0.001` |

**540 combinações possíveis; busca aleatória de 50** (seed 0). Fixos no bloco: `epochs=200`, `patience=20`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O gap treino–validação (`gap/rmse`) e a melhor época: o dropout atrasa a decoreba? Houve subajuste com 50%?

In [ ]:
rep.show_champion("lstm_b4_otimizacao")
rep.show_champion("lstm_b4_checagem")

In [ ]:
!python src/grid_search.py grids/lstm_b5_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b5_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b5_regularizacao", "triagem")

In [ ]:
rep.discarded_configs("lstm_b5_regularizacao")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b5_regularizacao`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b5_regularizacao")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b5_regularizacao", row="dropout", col="weight_decay")

In [ ]:
rep.heatmap("lstm_b5_regularizacao", row="dropout", col="weight_decay", value="gap/rmse_mean")

In [ ]:
rep.heatmap("lstm_b5_regularizacao", row="dropout", col="rnn_dropout")

In [ ]:
rep.param_effect("lstm_b5_regularizacao")

In [ ]:
rep.plot_grid_bars("lstm_b5_regularizacao")

#### Confirmação das finalistas e campeão — `lstm_b5_regularizacao`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b5_regularizacao", "final")

In [ ]:
rep.show_champion("lstm_b5_regularizacao")
rep.plot_finalists("lstm_b5_regularizacao")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b5_regularizacao"), "lstm_b5_regularizacao__*")

### 📝 Análise — Bloco 5

- **Dropout: 20% foi o melhor compromisso?** _…_
- **Dropout entre camadas e na entrada:** _…_
- **Decaimento dos pesos (L2):** _…_
- **Gap treino–validação e subajuste:** _…_
- **Ganho sobre o Bloco 4 (validação):** _…_

In [ ]:
backup("lstm_b5_regularizacao")

## Complementar A — Camadas e nós sob regularização

**Pergunta:** Com o dropout ajustado, redes com mais nós ou mais camadas voltam a compensar?

Checagem de interação entre decisões distantes no tempo: o Bloco 2 escolheu camadas e nós só com o dropout inicial. A lista diz que muitos nós *com regularização* podem aumentar a precisão, e este bloco testa isso. A combinação idêntica ao campeão do Bloco 5 é re-treinada e serve de **checagem de reprodutibilidade**.

**Base:** o melhor campeão entre `lstm_b5_regularizacao`.

| Eixo | Valores |
|---|---|
| `num_layers` | `1`, `2`, `3` |
| `hidden_size` | `32`, `64`, `128`, `256` |

**12 combinações.** A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds. Limite: 2,000,000 parâmetros.

**O que observar:** O heatmap camadas × nós, o gap, e se a repetição reproduz a validação do Bloco 5.

In [ ]:
rep.show_champion("lstm_b5_regularizacao")

In [ ]:
!python src/grid_search.py grids/lstm_b6_complementar_capacidade.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b6_complementar_capacidade`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b6_complementar_capacidade", "triagem")

In [ ]:
rep.discarded_configs("lstm_b6_complementar_capacidade")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b6_complementar_capacidade`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b6_complementar_capacidade")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b6_complementar_capacidade", row="hidden_size", col="num_layers")

In [ ]:
rep.heatmap("lstm_b6_complementar_capacidade", row="hidden_size", col="num_layers", value="gap/rmse_mean")

In [ ]:
rep.plot_grid_bars("lstm_b6_complementar_capacidade")

#### Confirmação das finalistas e campeão — `lstm_b6_complementar_capacidade`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b6_complementar_capacidade", "final")

In [ ]:
rep.show_champion("lstm_b6_complementar_capacidade")
rep.plot_finalists("lstm_b6_complementar_capacidade")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b6_complementar_capacidade"), "lstm_b6_complementar_capacidade__*")

### 📝 Análise — Complementar A

- **Com regularização, a capacidade ideal mudou?** _…_
- **A repetição reproduziu o campeão do Bloco 5? Qual o ruído entre execuções?** _…_

In [ ]:
backup("lstm_b6_complementar_capacidade")

## Complementar B — Número de épocas (early stopping)

**Pergunta:** O resultado está limitado pelo orçamento de épocas? Quanta paciência o early stopping deve ter?

O número de épocas é decidido pelo early stopping na `val/loss`, e os pesos da melhor época são restaurados. O limite sobe para 500 épocas e varia só a paciência. Com decaimento da lr, paciência longa também dá tempo para a lr cair. O campeão deste bloco é o **resultado principal** do estudo.

**Base:** o melhor campeão entre `lstm_b6_complementar_capacidade`.

| Eixo | Valores |
|---|---|
| `patience` | `5`, `10`, `20`, `40` |

**4 combinações.** Fixos no bloco: `epochs=500`. A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Se a `melhor_epoca_media` encosta no limite e se paciência maior melhora a validação ou só gasta GPU.

In [ ]:
rep.show_champion("lstm_b6_complementar_capacidade")

In [ ]:
!python src/grid_search.py grids/lstm_b7_complementar_epocas.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

In [ ]:
rep.discarded_configs("lstm_b7_complementar_epocas")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b7_complementar_epocas`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b7_complementar_epocas")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("lstm_b7_complementar_epocas")

#### Confirmação das finalistas e campeão — `lstm_b7_complementar_epocas`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b7_complementar_epocas", "final")

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")
rep.plot_finalists("lstm_b7_complementar_epocas")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b7_complementar_epocas"), "lstm_b7_complementar_epocas__*")

### 📝 Análise — Complementar B

- **Paciência maior ajudou? O resultado estava limitado pelas épocas?** _…_
- **Custo (tempo) × ganho:** _…_

In [ ]:
backup("lstm_b7_complementar_epocas")

## Bônus — LSTM × GRU

**Pergunta:** A célula LSTM é necessária, ou uma GRU (menos portas, menos parâmetros) chega ao mesmo resultado?

Fora da sequência principal: troca só a célula recorrente do campeão principal. A GRU usa a ativação tanh padrão. A receita campeã (LSTM) é re-treinada neste bloco como referência pareada.

**Base:** o melhor campeão entre `lstm_b7_complementar_epocas`.

| Eixo | Valores |
|---|---|
| `cell` | `lstm`, `gru` |

**2 combinações.** A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** A diferença pareada fold a fold, o número de parâmetros e o tempo de treino.

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")

In [ ]:
!python src/grid_search.py grids/lstm_b8_bonus_celula.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

In [ ]:
rep.discarded_configs("lstm_b8_bonus_celula")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b8_bonus_celula`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b8_bonus_celula")
display(importancia)
display(efeitos)

In [ ]:
rep.plot_grid_bars("lstm_b8_bonus_celula")

#### Confirmação das finalistas e campeão — `lstm_b8_bonus_celula`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b8_bonus_celula", "final")

In [ ]:
rep.show_champion("lstm_b8_bonus_celula")
rep.plot_finalists("lstm_b8_bonus_celula")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b8_bonus_celula"), "lstm_b8_bonus_celula__*")

### 📝 Análise — Bônus

- **GRU × LSTM (pareado):** _…_
- **Diferença de parâmetros e de tempo de treino:** _…_

In [ ]:
backup("lstm_b8_bonus_celula")

## Bônus — Data augmentation

**Pergunta:** Criar variações plausíveis das janelas de treino (ruído, amplitude, velocidade, ordem, recorte) melhora a previsão do campeão?

Data augmentation é **pré-processamento**, não hiperparâmetro da rede, por isso fica fora da sequência principal (como no estudo da CNN). A cada época, cada janela de treino é transformada com probabilidade 50%; validação e teste usam as janelas originais. As janelas estão normalizadas (z-score), então as intensidades são em desvios padrão. **Jittering:** ruído gaussiano (σ 0,03 fraca / 0,1 forte). **Scaling:** multiplica a janela por um fator ~ N(1, σ) (σ 0,1 / 0,2), simulando regimes de volatilidade; com alvo em retorno, o alvo é escalado junto. **Magnitude warping:** multiplica por uma curva suave aleatória (σ 0,1 / 0,2). **Time warping:** acelera e desacelera trechos da janela (σ 0,1 / 0,2), preservando o dia da decisão. **Permutation:** embaralha 3 ou 6 segmentos da janela; se não piorar, o modelo não está usando a ordem temporal. **Window slicing:** recorta 90% ou 70% da janela e reamostra para o tamanho original (a janela deslizante com sobreposição já é como as amostras são montadas). Também entra a combinação clássica jitter + scaling. **Base:** o campeão principal, re-treinado aqui sem augmentation como referência pareada, com mais épocas e paciência para todos, porque augmentation retarda a convergência. Em séries financeiras o ganho não é garantido; o bloco mede quais estratégias ajudam e quais atrapalham.

**Base:** o melhor campeão entre `lstm_b7_complementar_epocas`.

| Eixo | Valores |
|---|---|
| `augment` | `none`, `jitter`, `scaling`, `magwarp`, `timewarp`, `permutation`, `slicing`, `jitter+scaling` |
| `aug_strength` | `fraca`, `forte` |

**16 combinações.** Fixos no bloco: `epochs=300`, `patience=30`, `aug_prob=0.5`. A base é re-treinada sem mudanças (referência pareada). Triagem nos folds [3, 4, 5] de 5; as 3 melhores completam os 5 folds.

**O que observar:** O ganho pareado sobre a referência sem augmentation, a queda do gap treino–validação, e se a permutação piora (o que confirma que o LSTM usa a ordem temporal).

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")

In [ ]:
!python src/grid_search.py grids/lstm_b9_bonus_augmentation.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultados da triagem — `lstm_b9_bonus_augmentation`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). O teste não aparece aqui de propósito.

In [ ]:
rep.grid_ranking("lstm_b9_bonus_augmentation", "triagem")

In [ ]:
rep.discarded_configs("lstm_b9_bonus_augmentation")  # combinações descartadas antes do treino e o motivo

#### O que cada hiperparâmetro muda — `lstm_b9_bonus_augmentation`

Efeito de cada valor em relação ao valor da base, **controlando os outros eixos** (modelo aditivo sobre todas as configurações), e quanto da variação da métrica cada eixo explica. Verde = melhora além de 2× o ruído entre seeds; cinza = indistinguível de ruído.

In [ ]:
importancia, efeitos = rep.axis_importance("lstm_b9_bonus_augmentation")
display(importancia)
display(efeitos)

In [ ]:
rep.heatmap("lstm_b9_bonus_augmentation", row="augment", col="aug_strength")

In [ ]:
rep.heatmap("lstm_b9_bonus_augmentation", row="augment", col="aug_strength", value="gap/rmse_mean")

In [ ]:
rep.heatmap("lstm_b9_bonus_augmentation", row="augment", col="aug_strength", value="val/pocid_mean")

In [ ]:
rep.plot_grid_bars("lstm_b9_bonus_augmentation")

#### Confirmação das finalistas e campeão — `lstm_b9_bonus_augmentation`

As finalistas completaram todos os folds; o campeão é a melhor média da métrica de decisão nos K folds.

In [ ]:
rep.grid_ranking("lstm_b9_bonus_augmentation", "final")

In [ ]:
rep.show_champion("lstm_b9_bonus_augmentation")
rep.plot_finalists("lstm_b9_bonus_augmentation")

#### Comparação pareada por fold (validação)

Referência: a base do bloco re-treinada aqui (mesmos folds e seed); `veredito` compara o Δ com o ruído entre seeds.

In [ ]:
rep.paired_comparison(rep.base_config_name("lstm_b9_bonus_augmentation"), "lstm_b9_bonus_augmentation__*")

### 📝 Análise — Bônus

- **Alguma estratégia melhorou a validação além do ruído entre seeds?** _…_
- **Qual estratégia reduziu mais o gap treino–validação?** _…_
- **Permutação: o modelo depende da ordem temporal?** _…_
- **Time warping e slicing: distorcer o tempo ajuda ou atrapalha em retornos diários?** _…_
- **Intensidade fraca × forte:** _…_

In [ ]:
backup("lstm_b9_bonus_augmentation")

## Ablação — o que realmente importa no campeão

**Pergunta:** Das mudanças que levaram do LSTM padrão ao campeão, quais realmente melhoram o modelo e quais são dispensáveis?

Cada configuração parte do campeão principal e **desfaz uma única mudança**, voltando aquele hiperparâmetro ao valor do padrão do estudo. Todas rodam os K folds e são comparadas fold a fold com o campeão re-treinado aqui. Se piora ao desfazer (além do ruído entre seeds), a mudança ajuda; se fica dentro do ruído, é dispensável; se melhora ao desfazer, a mudança atrapalhava e só entrou por sorte na seleção. Hiperparâmetros cuja reversão não muda nada (ex.: momentum com Adam) são descartados como equivalentes. É a resposta mais direta para "o que melhora o modelo quando mexemos".

**Base:** o melhor campeão entre `lstm_b7_complementar_epocas`.

A base é re-treinada sem mudanças (referência pareada). Todas as configurações rodam os 5 folds.

**O que observar:** Quais mudanças ficam verdes (ajudam além do ruído) e se as mais importantes batem com a importância dos eixos em cada bloco.

In [ ]:
rep.show_champion("lstm_b7_complementar_epocas")

In [ ]:
!python src/grid_search.py grids/lstm_b10_ablacao.json --workers_per_gpu $WORKERS_PER_GPU --cpu_workers $CPU_WORKERS

#### Resultado da ablação — `lstm_b10_ablacao`

Δ = (campeão sem a mudança) − (campeão), fold a fold. Verde: a mudança ajuda além do ruído entre seeds; cinza: dispensável; vermelho: atrapalhava.

In [ ]:
rep.discarded_configs("lstm_b10_ablacao")  # mudanças cuja reversão não altera nada

In [ ]:
rep.ablation_table("lstm_b10_ablacao")

### 📝 Análise — Ablação

- **Quais mudanças realmente ajudam (além do ruído)?** _…_
- **Quais são dispensáveis e poderiam voltar ao padrão?** _…_
- **Alguma mudança atrapalhava?** _…_
- **Isso confirma a importância dos eixos vista em cada bloco?** _…_

In [ ]:
backup("lstm_b10_ablacao")

## O que realmente melhorou o modelo (validação)

Antes de revelar o teste: a **cadeia de decisões** reconstrói, pela herança entre blocos, o caminho do LSTM padrão até o
campeão principal (`lstm_b7_complementar_epocas`). Para cada passo: o que mudou, o Δ pareado por fold, em quantos folds a mudança venceu e se o
ganho passa do ruído entre seeds. Junto com a ablação (que desfaz cada mudança do campeão), mostra quais decisões de
hiperparâmetros realmente importam e quais só entraram por ruído.

In [ ]:
cadeia = rep.decision_chain("lstm_b7_complementar_epocas")
cadeia

In [ ]:
backup("cadeia")

## Resultado final — teste revelado

Até aqui todas as escolhas foram feitas pela validação walk-forward. Agora o período de teste é usado **uma única vez**,
para os campeões de cada bloco e para as referências. A tabela mostra média ± desvio do teste entre os K modelos (um por fold).
O campeão da sequência principal é o de `lstm_b7_complementar_epocas`; `lstm_b8_bonus_celula`, `lstm_b9_bonus_augmentation` mostra(m) eixos ortogonais sobre ele.
A concordância entre validação e teste é a evidência de que a metodologia não vazou informação; uma discrepância grande
(ex.: mudança de regime no período de teste) é um achado a investigar.

In [ ]:
final = rep.final_report(["lstm_b0_referencia", "lstm_b1_entrada", "lstm_b2_capacidade", "lstm_b3_ativacao_init", "lstm_b4_otimizacao", "lstm_b4_checagem", "lstm_b5_regularizacao", "lstm_b6_complementar_capacidade", "lstm_b7_complementar_epocas", "lstm_b8_bonus_celula", "lstm_b9_bonus_augmentation"])
final

Desempenho por série no teste: referências × campeão principal × bônus.

In [ ]:
campeao_principal = rep.champion_name("lstm_b7_complementar_epocas")
comparar = list(rep.results.configs("lstm_b0_referencia")["exp_name"]) + [campeao_principal] + [rep.champion_name(b) for b in ["lstm_b8_bonus_celula", "lstm_b9_bonus_augmentation"]]
rep.plot_per_ticker(comparar, metric="theil")
rep.plot_per_ticker(comparar, metric="pocid")

#### Métricas por série do campeão principal

Média ± desvio entre os K modelos (um por fold), no teste. Figura e CSV em `outputs/_relatorio/metricas_por_serie_test_<experimento>.*`.

In [ ]:
rep.champion_ticker_metrics("lstm_b7_complementar_epocas")

#### Previsão × real (ensemble dos K modelos)

In [ ]:
for t in data_mod.common.load_study()["dados"]["tickers"]:
    rep.plot_predictions(campeao_principal, ticker=t)

#### Valor econômico (ilustrativo)

Posição = sinal da previsão, sem custos de transação. **Não** é critério de decisão do estudo; só mostra se o sinal direcional tem valor além das métricas.

In [ ]:
rep.plot_strategy([e for e in rep.results.configs("lstm_b0_referencia")["exp_name"] if not e.endswith(("passeio_aleatorio", "media_historica"))] + [campeao_principal])

#### Resumo no W&B

Run `estudo__resumo`: cadeia de decisões, ruído entre seeds e o relatório final (o teste só é enviado agora, depois de revelado).

In [ ]:
import wandb_report
wandb_report.log_study("lstm_b7_complementar_epocas", ["lstm_b0_referencia", "lstm_b1_entrada", "lstm_b2_capacidade", "lstm_b3_ativacao_init", "lstm_b4_otimizacao", "lstm_b4_checagem", "lstm_b5_regularizacao", "lstm_b6_complementar_capacidade", "lstm_b7_complementar_epocas", "lstm_b8_bonus_celula", "lstm_b9_bonus_augmentation"])

In [ ]:
backup("final")

## 📝 Conclusões

- **O LSTM bateu o passeio aleatório no teste (theil < 1, skill > 0)? Por quanto, e de forma consistente entre folds e ações?** _…_
- **POCID: o modelo acerta a direção da cotação acima de 50%? Isso se converte em valor na estratégia ilustrativa?** _…_
- **Entrada (Bloco 1): janela, features e alvo, e por quê:** _…_
- **Camadas ocultas, nós e camada densa (Bloco 2): a regra prática (1–2 camadas, 5–10 unidades densas) se confirmou?** _…_
- **Ativações e inicialização (Bloco 3):** _…_
- **Otimização (Bloco 4): lr, decaimento (0,97?), momentum e batch:** _…_
- **A checagem de interação confirmou a escolha em blocos?** _…_
- **Dropout e decaimento dos pesos (Bloco 5): 20% foi o melhor compromisso?** _…_
- **Complementares: mais nós com regularização compensaram? O resultado estava limitado pelas épocas?** _…_
- **Bônus — LSTM × GRU:** _…_
- **Validação × teste concordaram? O período de teste (2022–2023) teve mudança de regime?** _…_